# 올리브영 상품 & 리뷰 크롤링

In [1]:
# 임포트
from urllib.parse import urlencode, urlparse, parse_qs, urlunparse
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
from bs4 import BeautifulSoup
import pandas as pd
import re

In [2]:
# 파라미터 설정
base_url = "https://www.oliveyoung.co.kr/store/search/getSearchMain.do"

params = {
    "query": "쿨링 샴푸",
    "giftYn": "N",
    "t_page": "쿨링샴푸검색",
    "t_click": "검색창",
    "t_search_name": "쿨링 샴푸",
}

per_page = 24

In [3]:
# url 만들기
def make_url(parsed, params, start_count):
    params["startCount"] = [str(start_count)]

    return urlunparse((
        parsed.scheme,
        parsed.netloc,
        parsed.path,
        parsed.params,
        urlencode(params, doseq=True),
        parsed.fragment,
    ))

In [4]:
# 상품 목록 파싱
def parse_products(html, page_no, seen_names):
    soup = BeautifulSoup(html, "html.parser")
    products = []
    item_index = 0

    for item in soup.select("li"):
        name_tag = item.select_one(".tx_name")
        price_tag = item.select_one(".tx_cur")
        brand_tag = item.select_one(".tx_brand")
        link_tag = item.select_one("a[href*='getGoodsDetail.do']")

        if not name_tag:
            continue

        item_index += 1
        review_rank = (page_no - 1) * per_page + item_index

        name = name_tag.get_text(" ", strip=True)

        if name in seen_names:
            continue

        product_url = None
        if link_tag:
            product_url = link_tag.get("href")
            if product_url.startswith("/"):
                product_url = "https://www.oliveyoung.co.kr" + product_url

        seen_names.add(name)

        products.append({
            "리뷰순위": review_rank,
            "페이지": page_no,
            "정렬": "리뷰 많은 순",
            "브랜드": brand_tag.get_text(strip=True) if brand_tag else None,
            "상품명": name,
            "가격": price_tag.get_text(strip=True) if price_tag else None,
            "상품URL": product_url,
        })

    return products

In [5]:
# 리뷰순 URL
async def get_review_sorted_url(page):
    first_url = base_url + "?" + urlencode(params)

    await page.goto(first_url, wait_until="domcontentloaded", timeout=60000)
    await page.wait_for_selector(".tx_name", timeout=30000)

    await page.get_by_text(re.compile("리뷰.*많은.*순")).click()
    await page.wait_for_timeout(3000)

    return page.url

In [6]:
# 전체 상품 수집
async def collect_products():
    all_products = []
    seen_names = set()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, slow_mo=100)
        page = await browser.new_page()

        review_url = await get_review_sorted_url(page)
        print("리뷰순 URL:", review_url)

        parsed = urlparse(review_url)
        review_params = parse_qs(parsed.query)

        page_no = 1

        while True:
            start_count = (page_no - 1) * per_page
            url = make_url(parsed, review_params, start_count)

            print(f"{page_no}페이지 수집 중...")

            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                await page.wait_for_selector(".tx_name", timeout=20000)
            except PlaywrightTimeoutError:
                print(f"{page_no}페이지에서 상품 없음. 종료!")
                break

            html = await page.content()
            page_products = parse_products(html, page_no, seen_names)

            if len(page_products) == 0:
                print(f"{page_no}페이지 신규 상품 0개. 종료!")
                break

            print(f"{page_no}페이지:", len(page_products), "개")
            all_products.extend(page_products)

            if len(page_products) < per_page:
                print("마지막 페이지 도달. 종료!")
                break

            page_no += 1

        await browser.close()

    return pd.DataFrame(all_products)

In [7]:
# 실행
df = await collect_products()
df.head()

리뷰순 URL: https://www.oliveyoung.co.kr/store/search/getSearchMain.do?startCount=0&sort=PRMUM_GDAS_TOT_CNT%2FDESC%2CWEIGHT%2FDESC%2CRANK%2FDESC&goods_sort=PRMUM_GDAS_TOT_CNT%2FDESC%2CWEIGHT%2FDESC%2CRANK%2FDESC&collection=ALL&realQuery=%EC%BF%A8%EB%A7%81+%EC%83%B4%ED%91%B8&reQuery=&viewtype=image&category=&catename=LCTG_ID&catedepth=1&rt=&setMinPrice=&setMaxPrice=&listnum=24&tmp_requery=&tmp_requery2=&categoryDepthValue=&cateId=&cateId2=&BenefitAll_CHECK=&query=%EC%BF%A8%EB%A7%81+%EC%83%B4%ED%91%B8&selectCateNm=%EC%A0%84%EC%B2%B4&firstTotalCount=43&typeChk=thum&branChk=&brandTop=&attrChk=&attrTop=&onlyOneBrand=&quickYn=N&cateChk=&benefitChk=&attrCheck0=&attrCheck1=&attrCheck2=&attrCheck3=&attrCheck4=&brandChkList=&benefitChkList=&_displayImgUploadUrl=https%3A%2F%2Fimage.oliveyoung.co.kr%2Fuploads%2Fimages%2Fdisplay%2F&recobellMbrNo=null&recobellCuid=8b47cf9f-efd1-48e4-8f83-10ee8a07945b&t_page=%EC%BF%A8%EB%A7%81%EC%83%B4%ED%91%B8%EA%B2%80%EC%83%89&t_click=%EC%83%81%ED%92%88%EB%B6%84%EB%A5

,리뷰순위,페이지,정렬,브랜드,상품명,가격,상품URL
0,1,1,리뷰 많은 순,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...
1,2,1,리뷰 많은 순,온더바디,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,"6,400원~",https://www.oliveyoung.co.kr/store/goods/getGo...
2,3,1,리뷰 많은 순,온더바디,[초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 ...,"5,000원~",https://www.oliveyoung.co.kr/store/goods/getGo...
3,4,1,리뷰 많은 순,라보에이치,[청량감MAX] 라보에이치 쿨링&노세범샴푸 750ML 기획(+샴푸100ML),"26,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...
4,5,1,리뷰 많은 순,클로란,[7월올영픽/지성두피&쿨링] 클로란 아쿠아민트 샴푸 더블 기획 (400ml 2입기획),"25,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...


In [8]:
# 확인
print(df.shape)
print(df.columns)

(43, 7)
Index(['리뷰순위', '페이지', '정렬', '브랜드', '상품명', '가격', '상품URL'], dtype='object')


In [9]:
df[["리뷰순위", "페이지", "브랜드", "상품명", "가격", "상품URL"]].head(10)

,리뷰순위,페이지,브랜드,상품명,가격,상품URL
0,1,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...
1,2,1,온더바디,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,"6,400원~",https://www.oliveyoung.co.kr/store/goods/getGo...
2,3,1,온더바디,[초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 ...,"5,000원~",https://www.oliveyoung.co.kr/store/goods/getGo...
3,4,1,라보에이치,[청량감MAX] 라보에이치 쿨링&노세범샴푸 750ML 기획(+샴푸100ML),"26,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...
4,5,1,클로란,[7월올영픽/지성두피&쿨링] 클로란 아쿠아민트 샴푸 더블 기획 (400ml 2입기획),"25,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...
5,6,1,클로란,[지성두피 & 쿨링] 클로란 아쿠아민트 딥클렌징 쿨링 샴푸 400ml,"16,900원",https://www.oliveyoung.co.kr/store/goods/getGo...
6,7,1,라보에이치,[청량감MAX] 라보에이치 쿨링&노세범샴푸 333ML 기획(+50ML 증정) 탈모증상완화,"14,900원",https://www.oliveyoung.co.kr/store/goods/getGo...
7,8,1,닥터포헤어,[7월 올영픽/한정기획] 닥터포헤어 에어로 쿨링 샴푸 750ml 기획 (+70ml*...,"26,900원",https://www.oliveyoung.co.kr/store/goods/getGo...
8,9,1,닥터포헤어,[여름필수템/두피쿨링] 닥터포헤어 에어로 쿨링 샴푸 400ml 기획 (+70ml),"17,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...
9,10,1,달리프,[쿨링감UP/탈모증상완화] 달리프 애플민트 쿨링 샴푸 500ml,"11,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...


In [ ]:
df.to_csv("data/oliveyoung_cooling_shampoo_products.csv", index=False, encoding="utf-8-sig")

# 리뷰 크롤링

In [ ]:
import pandas as pd
df = pd.read_csv("data/oliveyoung_cooling_shampoo_products.csv")
df.head()

,리뷰순위,페이지,정렬,브랜드,상품명,가격,상품URL
0,1,1,리뷰 많은 순,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...
1,2,1,리뷰 많은 순,온더바디,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,"6,400원~",https://www.oliveyoung.co.kr/store/goods/getGo...
2,3,1,리뷰 많은 순,온더바디,[초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 ...,"5,000원~",https://www.oliveyoung.co.kr/store/goods/getGo...
3,4,1,리뷰 많은 순,라보에이치,[청량감MAX] 라보에이치 쿨링&노세범샴푸 750ML 기획(+샴푸100ML),"26,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...
4,5,1,리뷰 많은 순,클로란,[7월올영픽/지성두피&쿨링] 클로란 아쿠아민트 샴푸 더블 기획 (400ml 2입기획),"25,900원~",https://www.oliveyoung.co.kr/store/goods/getGo...


In [12]:
from playwright.async_api import async_playwright
import pandas as pd
import re

max_reviews_per_product = 20

In [13]:
async def get_review_api_response(product_row):
    review_api_data = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, slow_mo=100)
        page = await browser.new_page()

        async def handle_response(response):
            if "review/api/v2/reviews/cursor" not in response.url:
                return

            try:
                data = await response.json()
                review_api_data.append(data)
                print("리뷰 API 잡음:", len(review_api_data))
            except Exception as e:
                print("리뷰 API JSON 읽기 실패:", e)

        page.on("response", handle_response)

        await page.goto(product_row["상품URL"], wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_timeout(3000)

        # 리뷰 탭 위치까지 스크롤
        for _ in range(8):
            await page.mouse.wheel(0, 1000)
            await page.wait_for_timeout(500)

        # 리뷰 탭 클릭
        review_tab = page.locator("[class*='GoodsDetailTabs_tab-item']", has_text=re.compile("리뷰")).first
        await review_tab.scroll_into_view_if_needed()
        await review_tab.click(force=True)

        await page.wait_for_timeout(5000)

        # 추가 리뷰 API 응답 유도
        for _ in range(15):
            await page.mouse.wheel(0, 1400)
            await page.wait_for_timeout(1000)

            # 2번 잡히면 보통 20개 확보
            if len(review_api_data) >= 2:
                await page.wait_for_timeout(1000)
                break

        await browser.close()

    return review_api_data

In [14]:
# 리뷰 API 파싱
def parse_review_api_data(api_data, product_row):
    rows = []
    seen_review_ids = set()

    for response_data in api_data:
        review_list = (
            response_data
            .get("data", {})
            .get("goodsReviewList", [])
        )

        for review in review_list:
            review_id = review.get("reviewId")

            if review_id in seen_review_ids:
                continue

            seen_review_ids.add(review_id)

            goods_dto = review.get("goodsDto", {})
            profile_dto = review.get("profileDto", {})

            rows.append({
                "리뷰순위": product_row["리뷰순위"],
                "브랜드": product_row["브랜드"],
                "상품명": product_row["상품명"],
                "가격": product_row["가격"],
                "상품URL": product_row["상품URL"],

                "상품내리뷰번호": len(rows) + 1,
                "리뷰ID": review_id,
                "리뷰내용": review.get("content"),
                "리뷰평점": review.get("reviewScore"),
                "리뷰타입": review.get("reviewType"),

                "리뷰어닉네임": profile_dto.get("memberNickname"),
                "도움돼요수": review.get("usefulPoint"),
                "사진여부": review.get("hasPhoto"),

                "옵션명": goods_dto.get("optionName"),
                "API상품명": goods_dto.get("goodsName"),
                "API상품번호": goods_dto.get("goodsNumber"),
            })

            if len(rows) >= max_reviews_per_product:
                return rows

    return rows

In [15]:
# 한 상품 테스트 
api_data = await get_review_api_response(df.iloc[0])
print("API 응답 수:", len(api_data))

review_rows = parse_review_api_data(api_data, df.iloc[0])
test_review_df = pd.DataFrame(review_rows)

test_review_df.shape

리뷰 API 잡음: 1
리뷰 API 잡음: 2
API 응답 수: 2


(20, 16)

In [16]:
# 확인
print(test_review_df.head())
print(test_review_df.shape)

   리뷰순위   브랜드                                             상품명       가격  \
0     1  온더바디  온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)  5,950원~   
1     1  온더바디  온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)  5,950원~   
2     1  온더바디  온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)  5,950원~   
3     1  온더바디  온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)  5,950원~   
4     1  온더바디  온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)  5,950원~   

                                               상품URL  상품내리뷰번호      리뷰ID  \
0  https://www.oliveyoung.co.kr/store/goods/getGo...        1  62073431   
1  https://www.oliveyoung.co.kr/store/goods/getGo...        2  61062952   
2  https://www.oliveyoung.co.kr/store/goods/getGo...        3  61757094   
3  https://www.oliveyoung.co.kr/store/goods/getGo...        4  61825678   
4  https://www.oliveyoung.co.kr/store/goods/getGo...        5  61971835   

                                                리뷰내용  리뷰평점    리뷰타입  \
0                곰팡이까지 없애주는 발을 씻자!

In [17]:
# 전체 상품 수집
async def collect_all_reviews(df, max_products=3):
    all_reviews = []

    target_df = df.head(max_products) if max_products else df

    for idx, product_row in target_df.iterrows():
        print(f"[{idx}] 리뷰 수집:", product_row["상품명"])

        try:
            api_data = await get_review_api_response(product_row)
            review_rows = parse_review_api_data(api_data, product_row)

            print("  API 응답 수:", len(api_data))
            print("  저장 리뷰 수:", len(review_rows))

            all_reviews.extend(review_rows)

        except Exception as e:
            print("  실패:", e)

    return pd.DataFrame(all_reviews)

In [18]:
# 테스트
review_df = await collect_all_reviews(df, max_products=3)
review_df.shape

[0] 리뷰 수집: 온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[1] 리뷰 수집: [NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아/레몬/쿨링/자몽)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[2] 리뷰 수집: [초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 5종 택1
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20


(60, 16)

In [19]:
review_df.head()

,리뷰순위,브랜드,상품명,가격,상품URL,상품내리뷰번호,리뷰ID,리뷰내용,리뷰평점,리뷰타입,리뷰어닉네임,도움돼요수,사진여부,옵션명,API상품명,API상품번호
0,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,1,62073431,곰팡이까지 없애주는 발을 씻자!! 떨어지면 무조건 쟁여야 합니다,5,NORMAL,글로리아ㅏ,7614.0,False,쿨링 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990
1,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,2,61062952,"여름 대비 필수템입니다. 발 전용 세정제라 처음에는 굳이 필요할까 싶었는데, 사용해...",5,NORMAL,applemint22,5310.0,True,쿨링 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990
2,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,3,61757094,🏷️사용감\r\n분사하자마자 상큼달달한 자몽껌 향이 화장실에 확 퍼집니다 인위적이지...,5,NORMAL,때구리,4500.0,True,레몬 1입,[초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 ...,A000000117399
3,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,4,61825678,비누체로 하기에는 귀찮아서 스프레이로 보드리면 간편해서 샀어요. 시원해요.,5,NORMAL,이얏후,4356.0,True,쿨링 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990
4,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,5,61971835,다른 향들 전부 사용해봤는데 뭐니뭐니해도 오리지널이 최고인 것 같습니다! 발체취가 ...,5,NORMAL,나건지마,3294.0,False,레몬 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990


In [20]:
review_df = await collect_all_reviews(df, max_products=None)
review_df.shape

[0] 리뷰 수집: 온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[1] 리뷰 수집: [NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아/레몬/쿨링/자몽)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[2] 리뷰 수집: [초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 5종 택1
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[3] 리뷰 수집: [청량감MAX] 라보에이치 쿨링&노세범샴푸 750ML 기획(+샴푸100ML)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[4] 리뷰 수집: [7월올영픽/지성두피&쿨링] 클로란 아쿠아민트 샴푸 더블 기획 (400ml 2입기획)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[5] 리뷰 수집: [지성두피 & 쿨링] 클로란 아쿠아민트 딥클렌징 쿨링 샴푸 400ml
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[6] 리뷰 수집: [청량감MAX] 라보에이치 쿨링&노세범샴푸 333ML 기획(+50ML 증정) 탈모증상완화
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[7] 리뷰 수집: [7월 올영픽/한정기획] 닥터포헤어 에어로 쿨링 샴푸 750ml 기획 (+70ml*2ea)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수: 2
  저장 리뷰 수: 20
[8] 리뷰 수집: [여름필수템/두피쿨링] 닥터포헤어 에어로 쿨링 샴푸 400ml 기획 (+70ml)
리뷰 API 잡음: 1
리뷰 API 잡음: 2
  API 응답 수

(753, 16)

In [21]:
review_df.head()

,리뷰순위,브랜드,상품명,가격,상품URL,상품내리뷰번호,리뷰ID,리뷰내용,리뷰평점,리뷰타입,리뷰어닉네임,도움돼요수,사진여부,옵션명,API상품명,API상품번호
0,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,1,62073431,곰팡이까지 없애주는 발을 씻자!! 떨어지면 무조건 쟁여야 합니다,5,NORMAL,글로리아ㅏ,7614.0,False,쿨링 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990
1,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,2,61062952,"여름 대비 필수템입니다. 발 전용 세정제라 처음에는 굳이 필요할까 싶었는데, 사용해...",5,NORMAL,applemint22,5310.0,True,쿨링 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990
2,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,3,61757094,🏷️사용감\r\n분사하자마자 상큼달달한 자몽껌 향이 화장실에 확 퍼집니다 인위적이지...,5,NORMAL,때구리,4500.0,True,레몬 1입,[초특가] 온더바디 발을씻자 코튼 풋샴푸 385ml 레몬/자몽/복숭아/쿨링/비누향 ...,A000000117399
3,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,4,61825678,비누체로 하기에는 귀찮아서 스프레이로 보드리면 간편해서 샀어요. 시원해요.,5,NORMAL,이얏후,4356.0,True,쿨링 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990
4,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",https://www.oliveyoung.co.kr/store/goods/getGo...,5,61971835,다른 향들 전부 사용해봤는데 뭐니뭐니해도 오리지널이 최고인 것 같습니다! 발체취가 ...,5,NORMAL,나건지마,3294.0,False,레몬 510ml,[NEW 2세대] 온더바디 발을씻자 코튼 풋샴푸 대용량 510ml 4종 택1(복숭아...,A000000165990


In [23]:
review_df.columns

Index(['리뷰순위', '브랜드', '상품명', '가격', '상품URL', '상품내리뷰번호', '리뷰ID', '리뷰내용', '리뷰평점',
       '리뷰타입', '리뷰어닉네임', '도움돼요수', '사진여부', '옵션명', 'API상품명', 'API상품번호'],
      dtype='object')

In [24]:
df.columns

Index(['리뷰순위', '페이지', '정렬', '브랜드', '상품명', '가격', '상품URL'], dtype='object')

In [29]:
print(df["상품명"].nunique())
print(df["상품URL"].nunique())
print(review_df["상품명"].nunique())
print(review_df["상품URL"].nunique())

43
43
40
40


In [ ]:
keep_cols = [
    "리뷰순위",
    "브랜드",
    "상품명",
    "가격",
    "상품내리뷰번호",
    "리뷰ID",
    "리뷰내용",
    "리뷰평점",
    "도움돼요수",
    "사진여부",
    "옵션명",
]

review_save_df = review_df[keep_cols].copy()

review_save_df.to_csv("data/oliveyoung_cooling_shampoo_reviews.csv", index=False, encoding="utf-8-sig")

In [ ]:
reviews_df = pd.read_csv('data/oliveyoung_cooling_shampoo_reviews.csv')
reviews_df

,리뷰순위,브랜드,상품명,가격,상품내리뷰번호,리뷰ID,리뷰내용,리뷰평점,도움돼요수,사진여부,옵션명
0,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",1,62073431,곰팡이까지 없애주는 발을 씻자!! 떨어지면 무조건 쟁여야 합니다,5,7614.00,False,쿨링 510ml
1,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",2,61062952,"여름 대비 필수템입니다. 발 전용 세정제라 처음에는 굳이 필요할까 싶었는데, 사용해...",5,5310.00,True,쿨링 510ml
2,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",3,61757094,🏷️사용감\r\n분사하자마자 상큼달달한 자몽껌 향이 화장실에 확 퍼집니다 인위적이지...,5,4500.00,True,레몬 1입
3,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",4,61825678,비누체로 하기에는 귀찮아서 스프레이로 보드리면 간편해서 샀어요. 시원해요.,5,4356.00,True,쿨링 510ml
4,1,온더바디,온더바디 발을씻자 코튼 풋샴푸 리필 500ml 4종 택1 (레몬/자몽/쿨링/비누향),"5,950원~",5,61971835,다른 향들 전부 사용해봤는데 뭐니뭐니해도 오리지널이 최고인 것 같습니다! 발체취가 ...,5,3294.00,False,레몬 510ml
...,...,...,...,...,...,...,...,...,...,...,...
748,43,필립비,필립비 페퍼민트 아보카도 샴푸 220ml (쿨링 두피샴푸),"70,000원",11,21778804,샴푸 이것저것 써보는거 좋아해요\r\n기대보다는 별로에요\r\n용량대비 가격은 많이...,3,4.50,False,
749,43,필립비,필립비 페퍼민트 아보카도 샴푸 220ml (쿨링 두피샴푸),"70,000원",12,22248664,"제가 쓰고 너무 좋아서, 지인 선물하려고 올영세일 마지막날에 급하게 구입했어요! 지...",5,2.25,False,
750,43,필립비,필립비 페퍼민트 아보카도 샴푸 220ml (쿨링 두피샴푸),"70,000원",13,22248647,"머리를 많이 써야하는 직업이기도 하고, 골프를 좋아해서.. 평소에 두피쿨링샴푸에 관...",5,2.25,False,
751,43,필립비,필립비 페퍼민트 아보카도 샴푸 220ml (쿨링 두피샴푸),"70,000원",14,29409906,향도 좋고 두피에 자극 없이 순해요. 다만 거품이 잘 안나는 편입니다,5,0.20,False,
